In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import pickle

In [2]:
###########################################################

notebook_dir = os.getcwd()

os.chdir(notebook_dir)

###########################################################

In [3]:
E = 1.0
lambdaE = 1.0
scE = 0.361711

tQ = 5.81537
t0 = 0.0370717

In [4]:
####################################################################################

try:
    with open('Fig_7_uMPS_data.pkl', 'rb') as file:
        M_data_dict = pickle.load(file)

except:
    print("Something went wrong")

####################################################################################

In [5]:
# Plotting style constants
ms = 7.5
fs = 20
lw = 0.75
tauQ_cut = 32

# Set up matplotlib backend and LaTeX styling
# NOTE: The 'pgf' backend requires a LaTeX installation (e.g., TeX Live, MiKTeX) on your machine.
plt.switch_backend('pgf')
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": fs,
})

h_targs = [0.000811, 0.01]
cmap = plt.cm.viridis # Added plt. prefix to ensure it resolves correctly
props = dict(boxstyle='round', facecolor='white', alpha=0.8)

# 1. Dynamically organize data
filtered_data = {h: [] for h in h_targs}
all_tauQ_vals = [] 

for keys, data_list in M_data_dict.items():
    h_val, tauQ = keys[0], keys[1]
    
    # Using np.isclose instead of exact matching to avoid floating point issues
    matched_h = next((h for h in h_targs if np.isclose(h_val, h, atol=1e-8)), None)
    
    if matched_h is not None and float(tauQ) <= tauQ_cut:
        filtered_data[matched_h].append((float(tauQ), data_list))
        all_tauQ_vals.append(float(tauQ))

# Create a single, global norm so the colorbar matches BOTH plots perfectly
if all_tauQ_vals:
    global_norm = mcolors.Normalize(vmin=min(all_tauQ_vals), vmax=max(all_tauQ_vals))
else:
    global_norm = mcolors.Normalize(vmin=1e-3, vmax=1)

# Initialize the plot
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(10, 10), sharex=True, sharey=True)
sub_labels = ['(a)', '(b)']

# 2. Loop dynamically over the subplots and targets
for i, (h_targ, ax, label) in enumerate(zip(h_targs, axs, sub_labels)):
    
    # Plot the specific lines for this h target
    for tauQ, data_list in filtered_data[h_targ]:
        t, Mz = data_list[0], data_list[1]
        
        line_color = cmap(global_norm(tauQ))
        t_c = round(scE * tauQ * tQ / t0, 6)
        that = np.sqrt(tauQ)
        
        t_c_index = np.argmin(np.abs(t - t_c))        

        ax.plot((t - t_c) / that, Mz / Mz[t_c_index], 
                color=line_color, linestyle='-', linewidth=lw, 
                rasterized=True) 
        
    # Vertical dashed lines and subplot styling
    ax.axvline(x=-1, color='black', linestyle='--', zorder=0)
    ax.axvline(x=1, color='black', linestyle='--', zorder=0)
    ax.set_ylabel(r'$M/M_c$')
    ax.set_xlim(-4, 4)
    ax.set_ylim(0.4, 1.8)
    
    # Annotations
    ax.text(0.025, 0.95, rf'$h = {h_targ}$', transform=ax.transAxes, 
            fontsize=fs-2, verticalalignment='top', bbox=props)
    ax.text(-0.06, 1.0, label, transform=ax.transAxes, 
            fontsize=fs, fontweight='bold', va='top', ha='right')

# Colorbar per subplot
sm = cm.ScalarMappable(cmap=cmap, norm=global_norm)
sm.set_array([]) 
# Passing axs (an array of axes) directly is often cleaner than axs.ravel().tolist() in newer Matplotlib versions
cbar = fig.colorbar(sm, ax=axs, orientation="horizontal", aspect=60, fraction=0.04, pad=0.12)
cbar.set_label(r'$\tau_Q$', fontsize=fs)

# Label the shared X axis on the bottom plot
axs[1].set_xlabel(r'$(t - t_c) / \hat{t}$')

plt.subplots_adjust(hspace=0.05, bottom=0.23)

# Save figure
filename = 'Fig_7.pdf'
plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.close()